#init

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim

In [0]:
#Read file from Bronze

In [0]:
df = spark.table('workspace.bronze.erp_cust_az12')

#Transformation

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))

##Customer ID cleaup

In [0]:
df = df.withColumn(
    'cid',
    F.when(col('cid').startswith('NAS'), 
        F.substring(col('cid'), 4, F.length(col('cid'))))
    .otherwise(col('cid'))
)

##Birthdate validation

In [0]:
df = df.withColumn(
    'BDATE', 
    F.when(col("BDATE") > F.current_date(), None)
    .otherwise(col("BDATE"))
)

##Gender Normalization

In [0]:
df = df.withColumn(
    'GEN',
    F.when(F.upper(col('GEN')).isin('F', 'FEMALE'), 'Female')
    .when(F.upper(col('GEN')).isin('M', 'MALE'), 'Male')
    .otherwise('n/a')
)

##Renaming Columns

In [0]:
rename_map = {
    'cid': 'customer_number',
    'BDATE': 'birth_date',
    'GEN': 'gender'
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

##rechecking dataframe

In [0]:
df.limit(10).display()

#Writing to Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('workspace.silver.erp_customers')

## rechecking silver table

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customers